# NTGEN Validation — screening generated nanotube candidates

Reads the candidates exported by `ntgen_generation.ipynb`
(`generated_cifs/*.cif` + `candidates_metadata.json`) and runs a progressive,
physics-motivated screening funnel, cheap filters first:

1. **Tier 1 — Geometry** (instant): min interatomic distance, radial-band
   compliance, wall thickness / wall count, cross-section ellipticity, coordination.
2. **Tier 2 — Chemistry** (instant): SMACT charge-neutrality + space-occupancy ratio.
3. **Tier 3 — CHGNet energy/forces** (GPU): total energy per atom, max|F|, magnetic moment.
4. **Tier 4 — Relaxation drift** (GPU, survivors only): energy drop + RMSD.
5. **Ranking + export**: composite score → `screening_results.csv`,
   `top_candidates_ranked.csv`, and the best CIFs.

> **Caveat.** The mp_20 checkpoint is out-of-distribution for ~42-atom
> vacuum-box tubes, so the Tier 3–4 numbers are **relative** screening signals
> across candidates, not absolute stability. Formation energy / energy-above-hull
> are intentionally omitted — the vacuum box distorts both.

**Colab:** Runtime → Change runtime type → T4 GPU, then Run all. Runs locally too
(auto-detects an existing checkout).

---
## 0. Setup — locate repo, install deps, load helpers

Reuses the project's existing stack only: `chgnet`, `pymatgen`, `ase`, `smact`
(plus numpy/scipy/scikit-learn). No new packages.

In [ ]:
import os, sys, subprocess, shutil

REPO_URL = 'https://github.com/3venthatguy/NTU-IQM.git'

def _find_repo(start):
    d = os.path.abspath(start)
    while True:
        if os.path.isdir(os.path.join(d, 'NTGENS')) and os.path.isdir(os.path.join(d, 'comp_models')):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            return None
        d = parent

# Use an existing checkout if we're inside one; else clone into /content (Colab).
REPO_DIR = _find_repo(os.getcwd())
if REPO_DIR is None:
    REPO_DIR = '/content/NTU-IQM'
    if not os.path.exists(os.path.join(REPO_DIR, '.git')):
        if os.path.exists(REPO_DIR):
            shutil.rmtree(REPO_DIR)
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR])

PROJECT_DIR  = os.path.join(REPO_DIR, 'NTGENS')
NOTEBOOK_DIR = os.path.join(REPO_DIR, 'comp_models', 'NTGEN_generation')
CIF_DIR      = os.path.join(NOTEBOOK_DIR, 'generated_cifs')

# scigen -> ntgent symlink self-heal (namespace parity for script/ imports)
if not os.path.exists(f'{PROJECT_DIR}/scigen') and os.path.exists(f'{PROJECT_DIR}/ntgent'):
    os.symlink('ntgent', f'{PROJECT_DIR}/scigen')
for p in (PROJECT_DIR, os.path.join(PROJECT_DIR, 'script')):
    if p not in sys.path:
        sys.path.insert(0, p)

# Dependencies (reuse the repo's stack; no new packages).
try:
    import chgnet, pymatgen, ase, smact  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'chgnet', 'pymatgen', 'ase', 'smact'])

print('Repo:     ', REPO_DIR)
print('CIF dir:  ', CIF_DIR,
      '(found)' if os.path.isdir(CIF_DIR) else '(MISSING — run ntgen_generation.ipynb Section 8 first)')

In [ ]:
import json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from sklearn.mixture import GaussianMixture
from pymatgen.core import Structure

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

# --- Geometry helpers (lifted verbatim from
#     comp_models/Analysis/nanotube_rtheta_forensics.ipynb) ---------------------
def detect_tube_axis(frac):
    "Lattice direction the atoms fill most (smallest circular vacuum gap in wrapped frac)."
    occ = []
    for k in range(3):
        f = np.sort(frac[:, k] % 1.0)
        if len(f) < 2:
            occ.append(0.0); continue
        gaps = np.diff(np.concatenate([f, [f[0] + 1.0]]))
        occ.append(1.0 - gaps.max())
    return int(np.argmax(occ))

def cylindrical_coords(cart, cell, axis):
    "Return r, theta, z_axial, u, v in the frame perpendicular to the tube axis."
    a_hat = cell[axis] / np.linalg.norm(cell[axis])
    z = cart @ a_hat
    perp = cart - np.outer(z, a_hat)
    other = [k for k in range(3) if k != axis]
    e1 = cell[other[0]] - (cell[other[0]] @ a_hat) * a_hat
    e1 = e1 / np.linalg.norm(e1)
    e2 = np.cross(a_hat, e1)
    ctr = perp.mean(0)
    u = (perp - ctr) @ e1
    v = (perp - ctr) @ e2
    return np.hypot(u, v), np.arctan2(v, u), z, u, v

def radial_density(r, L, nbins=30):
    "Jacobian-corrected radial number density rho(r)."
    if r.size < 2 or np.ptp(r) < 1e-6:
        return np.array([float(r.mean())]), np.array([float(r.size)]), np.array([r.size])
    nb = int(min(nbins, max(3, r.size)))
    counts, edges = np.histogram(r, bins=nb)
    ctr = 0.5 * (edges[:-1] + edges[1:]); dr = np.diff(edges)
    area = 2 * np.pi * ctr * dr
    rho = np.where(area > 0, counts / (area * L), 0.0)
    return ctr, rho, counts

def count_shells_bic(r, kmax=4):
    "Objective wall count: GMM over r, model chosen by minimum BIC."
    distinct = len(np.unique(np.round(r, 3)))
    kmax = min(kmax, distinct)
    if kmax <= 1:
        return 1
    r2 = r.reshape(-1, 1); best, bb = 1, np.inf
    for k in range(1, kmax + 1):
        b = GaussianMixture(k, random_state=0, n_init=1).fit(r2).bic(r2)
        if b < bb:
            bb, best = b, k
    return best

def ellipticity(u, v):
    cov = np.cov(np.vstack([u, v]))
    w = np.clip(np.linalg.eigvalsh(cov), 1e-9, None)
    return float(np.sqrt(1 - w[0] / w[1])), w

def nn_distances(cart, cell, axis, k=1):
    "Nearest-neighbour distance per atom, axial periodicity via +/-1 image tiling."
    imgs = np.vstack([cart + s * cell[axis] for s in (-1, 0, 1)])
    d, _ = cKDTree(imgs).query(cart, k=k + 1)
    return d[:, 1:]

def coordination(cart, cell, axis, scale=1.3):
    imgs = np.vstack([cart + s * cell[axis] for s in (-1, 0, 1)])
    tree = cKDTree(imgs)
    nn = np.median(nn_distances(cart, cell, axis))
    cut = scale * nn
    return np.array([len(tree.query_ball_point(p, cut)) - 1 for p in cart]), nn

def radii_in_frame(cart, ctr, a_hat, e1, e2):
    "Transverse r, u, v, z of each atom in the sampler's stored tube frame."
    z = cart @ a_hat
    rel = (cart - np.outer(z, a_hat)) - ctr
    u = rel @ e1; v = rel @ e2
    return np.hypot(u, v), u, v, z

print('geometry helpers ready')

---
## 1. Load candidates

Read every CIF in `generated_cifs/` and join the per-candidate geometry sidecar
(`candidates_metadata.json`) on the shared `stem` key. Candidates whose metadata
carries a real Alexandria band (`is_alx > 0`) get the exact sampler frame; the
rest fall back to a frame recomputed from their own atoms.

In [ ]:
meta_path = os.path.join(CIF_DIR, 'candidates_metadata.json')
assert os.path.isdir(CIF_DIR), (
    f'{CIF_DIR} missing — run ntgen_generation.ipynb Section 8 to export CIFs first.')

meta_by_stem = {}
if os.path.exists(meta_path):
    for rec in json.load(open(meta_path)):
        meta_by_stem[rec['stem']] = rec
else:
    print('WARNING: candidates_metadata.json not found — Tier-1 band checks will be skipped.')

candidates = []
for cif in sorted(glob.glob(os.path.join(CIF_DIR, '*.cif'))):
    stem = os.path.splitext(os.path.basename(cif))[0]
    try:
        s = Structure.from_file(cif)
    except Exception as e:
        print(f'  skip {stem}: {e}'); continue
    candidates.append({'stem': stem, 'structure': s, 'meta': meta_by_stem.get(stem)})

n_meta = sum(c['meta'] is not None for c in candidates)
print(f'Loaded {len(candidates)} candidates from {CIF_DIR}')
print(f'  {n_meta} carry geometry metadata (band checks on); {len(candidates) - n_meta} without')
assert candidates, 'No CIFs found — nothing to screen.'

---
## 2. Tier 1 — Geometry (instant, no GPU)

For each candidate we rebuild cartesian coordinates in the sampler's **raw**
frame (`frac @ lattice_raw`) so the exported tube frame stays aligned, then measure:

- **min interatomic distance** (axial-periodic nearest neighbour) — a hard floor
  (< 1.3 Å ⇒ overlapping atoms, reject);
- **radial-band compliance** — fraction of atoms inside the wall band
  `[r_min, r_max·(1+margin)]` (only for real-template `shl`; this is the direct
  "did the constraint mask work?" check);
- **wall thickness** (std of in-band r) and **wall count** (GMM+BIC);
- **cross-section ellipticity** and **median coordination number**.

In [ ]:
DMIN_FLOOR = 1.3   # Angstrom; below this, atoms overlap

t1 = []
for c in candidates:
    s, m = c['structure'], c['meta']
    cell = np.array(m['lattice_raw']) if (m and m.get('lattice_raw')) else s.lattice.matrix
    frac = s.frac_coords % 1.0
    cart = frac @ cell

    has_band = bool(m and float(m.get('is_alx', 0)) > 0 and m.get('tube_a_hat'))
    if has_band:
        axis = int(m['tube_axis'])
        ctr  = np.array(m['tube_centroid']); a_hat = np.array(m['tube_a_hat'])
        e1   = np.array(m['tube_e1']);       e2    = np.array(m['tube_e2'])
        r, u, v, z = radii_in_frame(cart, ctr, a_hat, e1, e2)
        r_lo = float(m['r_min']); r_hi = float(m['r_max']) * (1.0 + float(m.get('cyl_margin', 0.1)))
        in_band = (r >= r_lo) & (r <= r_hi)
        band_frac = float(np.mean(in_band))
        wall_std  = float(r[in_band].std()) if in_band.any() else np.nan
    else:
        axis = detect_tube_axis(frac)
        r, th, z, u, v = cylindrical_coords(cart, cell, axis)
        r_lo = r_hi = band_frac = wall_std = np.nan

    nn   = nn_distances(cart, cell, axis)
    dmin = float(nn.min()) if nn.size else np.nan
    walls = int(count_shells_bic(r))
    eps, _ = ellipticity(u, v)
    cn, _ = coordination(cart, cell, axis)

    row = {'stem': c['stem'], 'n': len(s), 'dmin': round(dmin, 3),
           'band_frac': (round(band_frac, 3) if band_frac == band_frac else np.nan),
           'wall_std': (round(wall_std, 3) if wall_std == wall_std else np.nan),
           'walls': walls, 'ellip': round(eps, 3), 'cn_med': float(np.median(cn)),
           'geom_ok': bool(dmin == dmin and dmin >= DMIN_FLOOR)}
    t1.append(row)
    c['_r'] = r; c['_band'] = (r_lo, r_hi)

t1_df = pd.DataFrame(t1)
display(t1_df)
print(f"geometry pass (dmin >= {DMIN_FLOOR} A): {int(t1_df['geom_ok'].sum())}/{len(t1_df)}")
print('band-compliance (real-template shl only): '
      f"mean {np.nanmean(t1_df['band_frac']):.0%}" if t1_df['band_frac'].notna().any()
      else 'band-compliance: n/a (no real-template shl candidates)')

In [ ]:
# Pooled radial distribution for the band candidates (mirrors the generation
# validation cell): atoms should pile up inside [r_lo, r_hi] and, with density
# guidance on, peak at the wall.
pooled = [(c['_r'], *c['_band']) for c in candidates
          if c.get('_band') and np.isfinite(c['_band'][0])]
if not pooled:
    print('No real-template band candidates to plot (van / fallback only).')
else:
    all_r = np.concatenate([r for r, _, _ in pooled])
    fig, ax = plt.subplots(figsize=(7.5, 4))
    ax.hist(all_r, bins=24, color='#43AA8B', edgecolor='white', density=True, label='all atoms')
    ax.axvline(np.median([hi for _, _, hi in pooled]), color='#F94144', ls='--', lw=2, label='median r_hi')
    ax.axvline(np.median([lo for _, lo, _ in pooled]), color='#277DA1', ls='--', lw=2, label='median r_lo')
    ax.set_xlabel('transverse radius r (A)'); ax.set_ylabel('density')
    ax.set_title('Tier 1: pooled radial distribution vs wall band')
    ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()

---
## 3. Tier 2 — Chemistry (instant)

Two cheap chemical-plausibility signals, ported from
`NTGENS/script/mat_utils.py` (`smact_validity` / `vol_density`) — reimplemented
inline to avoid that module's heavy imports:

- **SMACT validity** — can the composition be charge-balanced with a
  Pauling-electronegativity-consistent set of oxidation states (metals/alloys pass)?
- **Occupancy ratio** — packed atomic-sphere volume ÷ cell volume; the
  `< 1.7` gate (from `eval_screen.py`) flags implausibly dense cells.

Both are **soft** signals (they feed the score), not hard rejects.

In [ ]:
import itertools
from math import gcd
from functools import reduce
import smact
from smact.screening import pauling_test
from ase.data import covalent_radii, atomic_numbers

def smact_valid(structure, use_pauling=True, include_alloys=True):
    "Ported from mat_utils.smact_validity: charge-neutral + Pauling-consistent."
    comp = structure.composition.element_composition
    elems = tuple(str(e) for e in comp.elements)
    counts = [int(round(comp[e])) for e in comp.elements]
    if not counts:
        return False
    g = reduce(gcd, counts)
    counts = tuple(c // g for c in counts)
    if len(set(elems)) == 1:
        return True
    if include_alloys and all(el in smact.metals for el in elems):
        return True
    space = smact.element_dictionary(elems)
    smact_elems = [e[1] for e in space.items()]
    electronegs = [e.pauling_eneg for e in smact_elems]
    ox_combos = [e.oxidation_states for e in smact_elems]
    oxn = 1
    for oc in ox_combos:
        oxn *= max(1, len(oc))
    if oxn > 1e7:
        return False
    threshold = max(counts)
    for ox in itertools.product(*ox_combos):
        stoichs = [(c,) for c in counts]
        cn_e, _ = smact.neutral_ratios(ox, stoichs=stoichs, threshold=threshold)
        if cn_e:
            if use_pauling:
                try:
                    ok = pauling_test(ox, electronegs)
                except TypeError:
                    ok = True
            else:
                ok = True
            if ok:
                return True
    return False

def occupancy_ratio(structure):
    "Ported from mat_utils.vol_density: packed covalent-sphere volume / cell volume."
    tot = 0.0
    for site in structure:
        for el, occ in site.species.items():
            rad = covalent_radii[atomic_numbers[el.symbol]]
            tot += occ * (4.0 / 3.0) * np.pi * rad ** 3
    return tot / (abs(structure.volume) + 1e-4)

t2 = []
for c in candidates:
    s = c['structure']
    try:
        sv = bool(smact_valid(s))
    except Exception:
        sv = None
    occ = occupancy_ratio(s)
    t2.append({'stem': c['stem'], 'smact_valid': sv,
               'occupancy': round(occ, 3), 'occ_ok': bool(occ < 1.7)})

t2_df = pd.DataFrame(t2)
display(t2_df)
print('SMACT-valid:', int(sum(1 for r in t2 if r['smact_valid'])), '/', len(t2))

---
## 4. Tier 3 — CHGNet energy & forces (GPU)

CHGNet (universal MLIP) predicts total energy per atom, forces, and magnetic
moments — no DFT. We screen on **residual force** `max|F| < 0.1 eV/Å` (near a
local minimum) and keep energy/magmom as ranking signals. **No formation energy
or e-hull** (vacuum-box distorted).

In [ ]:
from chgnet.model.model import CHGNet
chgnet = CHGNet.load()

t3 = []
for c in candidates:
    s = c['structure']
    try:
        p = chgnet.predict_structure(s)
        f = np.asarray(p['f'])
        maxF = float(np.max(np.linalg.norm(f, axis=1)))
        m = p.get('m')
        mag = float(np.sum(np.abs(m)) / len(s)) if m is not None else 0.0
        E = float(p['e'])
    except Exception as e:
        print(f"  CHGNet failed on {c['stem']}: {e}")
        E = maxF = mag = np.nan
    t3.append({'stem': c['stem'], 'E_per_atom': round(E, 4),
               'max_force': round(maxF, 4), 'mag_per_atom': round(mag, 4),
               'force_ok': bool(maxF == maxF and maxF < 0.1)})

t3_df = pd.DataFrame(t3)
display(t3_df)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(t3_df['E_per_atom'].dropna(), bins=12, color='#42A5F5', edgecolor='white')
ax[0].set_xlabel('CHGNet energy (eV/atom)'); ax[0].set_ylabel('count')
ax[0].set_title('Total energy per atom (relative signal)')
ax[1].hist(t3_df['max_force'].dropna(), bins=12, color='#F8961E', edgecolor='white')
ax[1].axvline(0.1, color='#F94144', ls='--', lw=2, label='0.1 eV/A gate')
ax[1].set_xlabel('max |F| (eV/A)'); ax[1].set_ylabel('count')
ax[1].set_title('Residual force'); ax[1].legend()
plt.tight_layout(); plt.show()
print(f"force pass (max|F| < 0.1): {int(t3_df['force_ok'].sum())}/{len(t3_df)}")

---
## 5. Tier 4 — Relaxation drift (GPU, survivors only)

Relax the top survivors (passed the geometry floor, ranked by `max|F|`, capped at
~30%) with CHGNet and measure how far they move: **energy drop** and per-site
**RMSD**. Small drift ⇒ the generator already produced a near-equilibrium
geometry.

In [ ]:
from chgnet.model.dynamics import StructOptimizer
relaxer = StructOptimizer()

merged = t1_df.merge(t3_df, on='stem')
elig = merged[(merged['geom_ok']) & (merged['max_force'].notna())].sort_values('max_force')
n_relax = max(1, int(np.ceil(0.3 * len(elig)))) if len(elig) else 0
to_relax = elig.head(n_relax)['stem'].tolist()
print(f'Relaxing {len(to_relax)} / {len(candidates)} candidates (top ~30% by max|F|)...')

cand_by_stem = {c['stem']: c for c in candidates}
t4 = []
for stem in to_relax:
    s = cand_by_stem[stem]['structure']
    try:
        e_before = float(chgnet.predict_structure(s)['e'])
        res = relaxer.relax(s, verbose=False)
        relaxed = res['final_structure']
        e_after = float(res['trajectory'].energies[-1]) / len(relaxed)
        a, b = s.cart_coords, relaxed.cart_coords
        rmsd = float(np.sqrt(np.mean(np.sum((a - b) ** 2, axis=1)))) if len(a) == len(b) else np.nan
        t4.append({'stem': stem, 'e_drop': round(e_before - e_after, 4), 'rmsd': round(rmsd, 3)})
    except Exception as e:
        print(f'  relax failed on {stem}: {e}')
        t4.append({'stem': stem, 'e_drop': np.nan, 'rmsd': np.nan})

t4_df = pd.DataFrame(t4, columns=['stem', 'e_drop', 'rmsd'])
if len(t4_df):
    display(t4_df)
else:
    print('Nothing eligible to relax.')

---
## 6. Composite ranking & export

Combine the tiers into one score (low `max|F|`, SMACT-valid, occupancy-ok,
geometry-ok, high band-compliance, small relaxation RMSD), rank, and write
`screening_results.csv`, `top_candidates_ranked.csv`, and the best CIFs to
`screening_output/`.

In [ ]:
res = (t1_df.merge(t2_df, on='stem')
             .merge(t3_df, on='stem')
             .merge(t4_df, on='stem', how='left'))

def score_row(r):
    sc = 0.0
    mf = r['max_force'] if r['max_force'] == r['max_force'] else 1.0
    sc += 2.0 * (1.0 - np.tanh(mf / 0.1))          # low residual force
    sc += 1.0 * float(bool(r.get('smact_valid')))   # chemically balanceable
    sc += 0.5 * float(bool(r.get('occ_ok')))        # sane packing
    sc += 1.0 * float(bool(r.get('geom_ok')))       # no atom overlap
    if r['band_frac'] == r['band_frac']:            # tube-shape compliance
        sc += 1.5 * float(r['band_frac'])
    if 'rmsd' in r and r['rmsd'] == r['rmsd']:       # near-equilibrium
        sc += 1.0 * float(np.exp(-r['rmsd'] / 0.5))
    return round(sc, 3)

res['score'] = res.apply(score_row, axis=1)
res = res.sort_values('score', ascending=False).reset_index(drop=True)

out_dir = os.path.join(NOTEBOOK_DIR, 'screening_output')
top_dir = os.path.join(out_dir, 'top_candidates')
os.makedirs(top_dir, exist_ok=True)
res.to_csv(os.path.join(out_dir, 'screening_results.csv'), index=False)

N = min(10, len(res))
res.head(20).to_csv(os.path.join(out_dir, 'top_candidates_ranked.csv'), index=False)
for stem in res.head(N)['stem']:
    src = os.path.join(CIF_DIR, stem + '.cif')
    if os.path.exists(src):
        shutil.copy(src, top_dir)

print(f'Wrote screening_results.csv ({len(res)} rows), top_candidates_ranked.csv, '
      f'and {N} CIFs -> {out_dir}')
display(res.head(N))

---
## Notes

- **Relative, not absolute.** With the mp_20 checkpoint OOD for ~42-atom vacuum
  tubes, Tier 3–4 rank candidates against **each other**; they are not a verdict
  on synthesizability. Promising survivors still need DFT.
- **Why no e-hull / formation energy.** The vacuum box inflates the cell volume
  and total energy in ways that break both quantities for a 1D tube; residual
  forces and relaxation drift are the vacuum-robust stability signals.
- **Band compliance is the key generation-quality signal.** If real-template
  `shl` candidates show low `band_frac`, revisit `CYL_MASKING` / `DENSITY_MASKING`
  in `ntgen_generation.ipynb` — the soft radial mask under-confined the atoms.
- **Next:** feed `screening_output/top_candidates/` into DFT or the GNN screener
  (`NTGENS/script/eval_screen.py`) for a heavier second pass.